# WireMod detector unisim (Products A + B)

Match at **`sel_all`**, then for **each universe** (calo ± and efield) walk the
full selection pipeline on matched files:

1. **Product A** — selection / cut-stage variables
2. **Product B** — measurement variables at final (`2prong-mup`) stage

Per geometry (YZ, XTXW), the **final** WireMod unisim is the **total envelope**:
max |shift| over all `ccal/alpha/beta/R` ± universes **and** `efield` vs the
in-file `cv` universe.

**Inputs** (Sep-14 WireMod + Sep-4 CV):

| Label | Campaign |
|---|---|
| YZ | `2026_09_14_025216__sel_all-mc-BNB_cosmics-WireModYZ` |
| XTXW | `2026_09_14_024629__sel_all-mc-BNB_cosmics-WireModXTXW` |
| CV | `2026_09_04_172912__sel_all-mc-CV` |

Matched products: `WireMod/matched/{yz,xtxw,cv}/*_matched.df`
(common keys ≈ 5.33M events). **CV** is POT / match reference only — no envelopes
from CV. Envelopes come from YZ / XTXW vs their in-file `cv`.

Batched driver: `scripts/reprocess_wiremod.py` (batch hist walk + RSS limit).

**Final outputs**:
- `WireMod/DetectorSelection/detector_sel_syst_dict.npz` (A)
- `WireMod/Detector/detector_syst_dict.npz` (B)


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, SPRING_GEN1_ROOT
from analysis_village.numucc_1p0pi.scripts import dent_compare as dc
from analysis_village.numucc_1p0pi.syst_disk_layout import (
    FILE_DETECTOR,
    FILE_DETECTOR_SEL,
    SUB_DETECTOR,
    SUB_DETECTOR_SEL,
)
from analysis_village.numucc_1p0pi.syst_detvar_common import (
    WIREMOD_ENVELOPE_SHIFTED,
    accumulate_wiremod_matched_products,
    assert_variations_matched,
    assert_wiremod_universes,
    build_wiremod_detector_dict,
    frac_unc_pct_from_pack,
    glob_matched_dfs,
    load_or_build_cache,
    log,
    max_envelope_univ_counts,
    pot_scales_to_cv,
    save_detector_npz,
    sum_pot_from_matched_files,
    wiremod_component_shifted_univs,
    wiremod_geometry_hists_for_envelope,
)
from analysis_village.numucc_1p0pi.utils import dpi
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig


In [ ]:
DFS = Path(os.environ.get("NUMUCC_SPRING_GEN1_ROOT", SPRING_GEN1_ROOT))

OUT_BASE = Path(os.environ.get("WIREMOD_OUT_BASE", str(PLOTS_BASE / "systematics-final" / "WireMod")))
_MATCHED = {
    "YZ": OUT_BASE / "matched" / "yz",
    "XTXW": OUT_BASE / "matched" / "xtxw",
    "CV": OUT_BASE / "matched" / "cv",
}
_RAW = {
    "YZ": DFS / os.environ.get(
        "WIREMOD_YZ_SEL_ALL", "2026_09_14_025216__sel_all-mc-BNB_cosmics-WireModYZ"
    ),
    "XTXW": DFS / os.environ.get(
        "WIREMOD_XTXW_SEL_ALL", "2026_09_14_024629__sel_all-mc-BNB_cosmics-WireModXTXW"
    ),
    "CV": DFS / os.environ.get("WIREMOD_CV_SEL_ALL", "2026_09_04_172912__sel_all-mc-CV"),
}

if all(p.is_dir() and any(p.glob("*_matched.df")) for p in _MATCHED.values()):
    WIREMOD_DIRS = _MATCHED
    print("using WireMod/matched/ (*_matched.df)")
else:
    WIREMOD_DIRS = _RAW
    print("using raw sel_all dirs — run scripts/reprocess_wiremod.py (or match write) first")

CACHE_DIR = OUT_BASE / "cache"
FIG_DIR = OUT_BASE / "plots"
DET_B_DIR = OUT_BASE / SUB_DETECTOR
DET_A_DIR = OUT_BASE / SUB_DETECTOR_SEL
INSPECT_DIR = OUT_BASE / "inspect_envelopes"
INSPECT_NPZ_DIR = INSPECT_DIR / "npz"
INSPECT_FIG_DIR = INSPECT_DIR / "plots"

CACHE_PATH = CACHE_DIR / "wiremod_sel_all_products.pkl"
NPZ_B = DET_B_DIR / FILE_DETECTOR
NPZ_A = DET_A_DIR / FILE_DETECTOR_SEL

FORCE_REPROCESS = os.environ.get("WIREMOD_FORCE_REPROCESS", "0") == "1"
ASSERT_MAX_FILES = int(os.environ.get("WIREMOD_ASSERT_MAX_FILES", "20"))
SAVE_FIGS = True

for d in (CACHE_DIR, FIG_DIR, DET_A_DIR, DET_B_DIR, INSPECT_NPZ_DIR, INSPECT_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

VARIATIONS = {
    lab: glob_matched_dfs(p, filename_str="sel_all")
    for lab, p in WIREMOD_DIRS.items()
    if Path(p).is_dir()
}
for lab, files in VARIATIONS.items():
    print(f"{lab}: {len(files)} matched sel_all files  ({WIREMOD_DIRS[lab]})")
print("cache:", CACHE_PATH)
print("Product A (final):", NPZ_A)
print("Product B (final):", NPZ_B)
print("inspect only:", INSPECT_DIR)
print("total envelope univs:", WIREMOD_ENVELOPE_SHIFTED)


In [ ]:
assert_variations_matched(
    {k: v for k, v in VARIATIONS.items() if v},
    max_files_per_var=ASSERT_MAX_FILES,
    format="sel_all",
)
# Calo ± + efield only on WireMod geometries — CV is reference only (no envelopes)
for lab, files in VARIATIONS.items():
    if not files or lab == "CV":
        continue
    log(f"checking WireMod universes (calo + efield): {lab}")
    assert_wiremod_universes(files, require_efield=True)


In [ ]:
FINAL_VAR_DEFS = dc.build_final_var_defs()
print(f"final vars: {len(FINAL_VAR_DEFS)}")


def _build_wiremod_cache():
    by_geom = {}
    pot_by = {}
    for lab, files in VARIATIONS.items():
        if not files:
            continue
        if lab == "CV":
            pot = sum_pot_from_matched_files(files)
            pot_by[lab] = pot
            log(f"[{lab}] POT reference only ({len(files)} matched files) POT={pot:.3e}")
            continue
        log(f"[{lab}] walk {len(files)} matched sel_all+calo+efield files …")
        prod = accumulate_wiremod_matched_products(
            files,
            final_var_defs=FINAL_VAR_DEFS,
            include_cut_stage=True,
        )
        by_geom[lab] = prod
        pot_by[lab] = prod["pot"]
        log(f"  {lab} POT={prod['pot']:.3e} univs={prod['universes']}")
    pot_scales = pot_scales_to_cv(pot_by, reference="CV") if "CV" in pot_by else {k: 1.0 for k in pot_by}
    for lab, sc in pot_scales.items():
        if lab not in by_geom or abs(float(sc) - 1.0) < 1e-15:
            continue
        sc = float(sc)
        for univ, payload in by_geom[lab]["by_universe"].items():
            for key in ("hists_cut", "hists_final"):
                payload[key] = {k: np.asarray(v, dtype=float) * sc for k, v in payload[key].items()}
    return {
        "by_geom": by_geom,
        "pot_by_variation": pot_by,
        "pot_scales": pot_scales,
        "match_stage": "sel_all",
        "envelope": "calo_plus_efield",
        "cv_role": "pot_reference_only",
        "variations": {k: list(v) for k, v in VARIATIONS.items()},
    }


payload = load_or_build_cache(CACHE_PATH, _build_wiremod_cache, force=FORCE_REPROCESS)
by_geom = payload["by_geom"]
print("POT scales:", payload.get("pot_scales"))
for lab, prod in by_geom.items():
    print(lab, "cut vars", len(prod["cut_var_names"]), "final vars", len(prod["final_var_names"]), "univs", prod.get("universes"))


## Final WireMod products (calo + efield total envelope)

These NPZs under `Detector/` / `DetectorSelection/` are what
`systematics-detector.ipynb` and `systematics-summary.ipynb` load.


In [ ]:
def _all_hists_for_product(product: str):
    out = {}
    for lab, prod in by_geom.items():
        out[lab] = wiremod_geometry_hists_for_envelope(prod["by_universe"], product=product)
    return out


all_hists_a = _all_hists_for_product("cut")
all_hists_b = _all_hists_for_product("final")

cut_names = next(iter(by_geom.values()))["cut_var_names"]
final_names = next(iter(by_geom.values()))["final_var_names"]

dict_a = build_wiremod_detector_dict(
    all_hists_a, cut_names, wiremod_labels=("YZ", "XTXW"), shifted_univs=WIREMOD_ENVELOPE_SHIFTED
)
dict_b = build_wiremod_detector_dict(
    all_hists_b, final_names, wiremod_labels=("YZ", "XTXW"), shifted_univs=WIREMOD_ENVELOPE_SHIFTED
)

save_detector_npz(
    dict_a,
    NPZ_A,
    manifest={
        "source": "WireMod",
        "product": "A_selection",
        "match_stage": "sel_all",
        "method": "total_envelope_calo_plus_efield",
        "shifted_univs": list(WIREMOD_ENVELOPE_SHIFTED),
        "cache": str(CACHE_PATH),
        "n_vars": len(dict_a.get("detector", {})),
    },
)
save_detector_npz(
    dict_b,
    NPZ_B,
    manifest={
        "source": "WireMod",
        "product": "B_measurement",
        "match_stage": "sel_all",
        "method": "total_envelope_calo_plus_efield",
        "shifted_univs": list(WIREMOD_ENVELOPE_SHIFTED),
        "cache": str(CACHE_PATH),
        "n_vars": len(dict_b.get("detector", {})),
    },
)
print("Product A vars:", len(dict_a.get("detector", {})))
print("Product B vars:", len(dict_b.get("detector", {})))


## Rate + envelope + uncertainty plots (Products A & B)

Dent-style layout: **rate** (CV + YZ/XTXW envelope bands), **env/CV ratio**,
and **fractional uncertainty [%]** for each WireMod geometry.


In [ ]:
def _envelope_lo_hi(hists, var_name, shifted_univs=WIREMOD_ENVELOPE_SHIFTED):
    n_cv = np.asarray(hists["cv"][var_name], dtype=float)
    keys = [u for u in shifted_univs if u in hists and u != "cv" and var_name in hists[u]]
    if not keys:
        return n_cv, n_cv.copy(), n_cv.copy()
    stacked = np.stack([np.asarray(hists[u][var_name], dtype=float) for u in keys], axis=0)
    return n_cv, stacked.min(axis=0), stacked.max(axis=0)


def _plot_wiremod_envelope_compare(
    vsn: str,
    all_hists: dict,
    bins,
    xlabel: str,
    dict_det: dict,
    out_dir: Path,
    *,
    tag: str,
    title: str | None = None,
):
    bins = np.asarray(bins, dtype=float)
    centers = 0.5 * (bins[:-1] + bins[1:])
    fig = plt.figure(figsize=(11.0, 5.2), layout="constrained")
    gs = fig.add_gridspec(2, 2, height_ratios=[2.2, 1.0], hspace=0.05, wspace=0.22)
    ax_rate = fig.add_subplot(gs[0, 0])
    ax_ratio = fig.add_subplot(gs[1, 0], sharex=ax_rate)
    ax_unc = fig.add_subplot(gs[0, 1])

    geom_styles = (
        ("YZ", "wiremod_yz", "C0", 0.28),
        ("XTXW", "wiremod_xtxw", "C1", 0.22),
    )
    n_cv_ref = None
    for lab, tag_key, color, alpha in geom_styles:
        if lab not in all_hists or "cv" not in all_hists[lab] or vsn not in all_hists[lab]["cv"]:
            continue
        n_cv, lo, hi = _envelope_lo_hi(all_hists[lab], vsn)
        if n_cv_ref is None:
            n_cv_ref = n_cv
            ax_rate.hist(centers, bins=bins, weights=n_cv, histtype="step", lw=1.8, color="black", label="CV")
        ax_rate.fill_between(centers, lo, hi, step="mid", color=color, alpha=alpha, label=f"{lab} envelope")
        # ratio of envelope extreme vs CV
        n_env = max_envelope_univ_counts(n_cv, all_hists[lab], vsn, WIREMOD_ENVELOPE_SHIFTED)
        ratio = np.full_like(n_cv, np.nan, dtype=float)
        ok = n_cv > 0
        ratio[ok] = n_env[ok] / n_cv[ok]
        valid = np.isfinite(ratio)
        if np.any(valid):
            ax_ratio.plot(centers[valid], ratio[valid], drawstyle="steps-mid", lw=1.6, color=color, label=lab)
        pack = dict_det.get(f"detector-{tag_key}", {}).get(vsn)
        if pack is not None:
            ax_unc.hist(
                centers, bins=bins, weights=frac_unc_pct_from_pack(pack),
                histtype="step", lw=1.8, color=color, label=lab,
            )

    ax_rate.set_ylabel("Events")
    ax_rate.legend(fontsize=8)
    ax_rate.grid(True, alpha=0.3)
    ax_rate.tick_params(labelbottom=False)
    if len(centers) == 1:
        ax_rate.set_xlim(bins[0], bins[-1])

    ax_ratio.axhline(1.0, color="black", ls="--", lw=1.0)
    ax_ratio.set_ylabel("env / CV")
    ax_ratio.set_xlabel(xlabel)
    ax_ratio.legend(fontsize=8)
    ax_ratio.grid(True, alpha=0.3)

    ax_unc.set_ylabel("Uncertainty [%]")
    ax_unc.set_xlabel(xlabel)
    ax_unc.set_ylim(bottom=0)
    ax_unc.legend(fontsize=8)
    ax_unc.grid(True, alpha=0.3)
    if len(centers) == 1:
        ax_unc.set_xlim(bins[0], bins[-1])

    fig.suptitle(title if title is not None else vsn, fontsize=11)
    if SAVE_FIGS:
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / f"wiremod_{tag}__{vsn}.png", dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close(fig)


FIG_B_DIR = FIG_DIR / "product_B_final"
FIG_A_DIR = FIG_DIR / "product_A_selection"
CUT_VAR_DEFS = dc.build_sel_all_var_defs()

_n_b = 0
for vsn, cfg in FINAL_VAR_DEFS.items():
    _plot_wiremod_envelope_compare(
        vsn, all_hists_b, cfg["bins"], cfg.get("label", vsn), dict_b, FIG_B_DIR, tag="final",
    )
    _n_b += 1
print(f"Product B plots: {_n_b} → {FIG_B_DIR}")

_n_a = 0
for vsn, cfg in CUT_VAR_DEFS.items():
    _plot_wiremod_envelope_compare(
        vsn, all_hists_a, cfg["bins"], cfg.get("label", vsn), dict_a, FIG_A_DIR,
        tag="cut", title=cfg.get("stage_key", "") or vsn,
    )
    _n_a += 1
print(f"Product A plots: {_n_a} → {FIG_A_DIR}")


## Inspect — component envelopes (diagnostic only)


In [ ]:
COMPONENT_SHIFTED = wiremod_component_shifted_univs()
print("component envelopes:", {k: list(v) for k, v in COMPONENT_SHIFTED.items()})

inspect_dicts_b = {}
inspect_summary = {"product": "B_measurement", "note": "inspection only", "components": {}}

for comp, shifted in COMPONENT_SHIFTED.items():
    d = build_wiremod_detector_dict(
        all_hists_b, final_names, wiremod_labels=("YZ", "XTXW"), shifted_univs=shifted,
    )
    inspect_dicts_b[comp] = d
    out_npz = INSPECT_NPZ_DIR / f"wiremod_envelope_{comp}_productB.npz"
    save_detector_npz(
        d, out_npz,
        manifest={
            "source": "WireMod", "product": "B_measurement_inspect", "component": comp,
            "method": "component_envelope", "shifted_univs": list(shifted),
            "not_for_downstream": True, "cache": str(CACHE_PATH),
        },
    )
    row = {"shifted_univs": list(shifted)}
    for key in ("detector-wiremod_yz", "detector-wiremod_xtxw", "detector"):
        pack = d.get(key, {}).get("integrated")
        if pack is None:
            continue
        w = frac_unc_pct_from_pack(pack)
        row[key] = float(np.asarray(w).ravel()[0]) if len(np.asarray(w).ravel()) else float("nan")
    inspect_summary["components"][comp] = row
    print(f"  {comp}: wrote {out_npz.name}  integrated={row}")

summary_path = INSPECT_DIR / "component_envelope_summary.json"
summary_path.write_text(json.dumps(inspect_summary, indent=2))
print("wrote", summary_path)
